In [12]:
import numpy as np
import scipy
import pandas as pd
import cv2
from matplotlib import pyplot as plt

In [14]:
# Load two poses of an image
img1 = cv2.imread("./pose1.png")
img2 = cv2.imread("./pose2.png")

# Flip images upside down 
img1 = cv2.flip(img1, 0)
img2 = cv2.flip(img2, 0)

# Resize images to fit on screen
scale = 0.33
img1 = cv2.resize(img1, (int(img1.shape[1] * scale), int(img1.shape[0] * scale)))
img2 = cv2.resize(img2, (int(img2.shape[1] * scale), int(img2.shape[0] * scale)))

combined = np.hstack((img1, img2))

cv2.imshow("Two-view", combined)
cv2.waitKey(2000)
cv2.destroyAllWindows()

In [15]:
# Convert the loaded image to grayscale
gray_img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray_img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

combined_gray = np.hstack((gray_img1, gray_img2))

cv2.imshow("Two-view Gray", combined_gray)
cv2.waitKey(2000)
cv2.destroyAllWindows()

# Extrinsic Calibration

## Salient Point Matching

### Salient Points Detection

In [18]:
# SIFT
sift = cv2.SIFT_create()

kp1 = sift.detect(gray_img1,None)
kp2 = sift.detect(gray_img2,None)
 
sift_img1=cv2.drawKeypoints(gray_img1,kp1,img1)
sift_img2=cv2.drawKeypoints(gray_img2,kp2,img2)

combined_sift = np.hstack((sift_img1, sift_img2))

cv2.imshow("SIFT points", combined_sift)
cv2.waitKey(2000)
cv2.destroyAllWindows()

In [19]:
# ORB
orb = cv2.ORB_create()

kp1 = orb.detect(gray_img1,None)
kp2 = orb.detect(gray_img2,None)

orb_img1 = cv2.drawKeypoints(gray_img1,kp1,img1)
orb_img2 = cv2.drawKeypoints(gray_img2,kp2,img2)

combined_orb = np.hstack((orb_img1, orb_img2))

cv2.imshow("ORB points", combined_orb)
cv2.waitKey(2000)
cv2.destroyAllWindows()

In [ ]:
# SURF
# Note: SURF is patented and may not be available in all OpenCV installations

#surf = cv2.xfeatures2d.SURF_create()

#kp1 = surf.detect(gray_img1,None)
#kp2 = surf.detect(gray_img2,None)

#surf_img1 = cv2.drawKeypoints(gray_img1,kp1,img1)
#surf_img2 = cv2.drawKeypoints(gray_img2,kp2,img2)

#combined_surf = np.hstack((surf_img1, surf_img2))

#cv2.imshow("SURF points", combined_surf)
#cv2.waitKey(10000)
#cv2.destroyAllWindows()


In [20]:
# AKAZE
akaze = cv2.AKAZE_create()

kp1 = akaze.detect(gray_img1,None)
kp2 = akaze.detect(gray_img2,None)

akaze_img1 = cv2.drawKeypoints(gray_img1,kp1,img1)
akaze_img2 = cv2.drawKeypoints(gray_img2,kp2,img2)

combined_akaze = np.hstack((akaze_img1, akaze_img2))

cv2.imshow("AKAZE points", combined_akaze)
cv2.waitKey(2000)
cv2.destroyAllWindows()

In [21]:
# Define the model to use for keypoint description
model = cv2.SIFT_create()

kp1, des1 = model.detectAndCompute(gray_img1, None)
kp2, des2 = model.detectAndCompute(gray_img2, None)

### Salient Points Matching

In [26]:
# BRUTE-FORCE
bf = cv2.BFMatcher()

bf_matches = bf.knnMatch(des1, des2, k=2)

# Lowe's ratio test
good_matches = []
for m,n in bf_matches:
    if m.distance < 0.75 * n.distance:
        good_matches.append(m)
        
bf_matched_img = cv2.drawMatches(img1, kp1, img2, kp2, good_matches, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

cv2.imshow("Brute Force Matched Points", bf_matched_img)
cv2.waitKey(10000)
cv2.destroyAllWindows()

In [33]:
# FLANN

flann_params = dict(algorithm=1, trees=5)
search_params = dict(checks=50)

flann = cv2.FlannBasedMatcher(flann_params, search_params)

#flann_matches = flann.knnMatch(des1, des2, k=2)

matches_1to2 = flann.knnMatch(des1, des2, k=2) 
matches_2to1 = flann.knnMatch(des2, des1, k=2)

# Lowe's ratio test
good_matches_1to2 = []
for m,n in matches_1to2:
    if m.distance < 0.9 * n.distance:
        good_matches_1to2.append(m)
        
good_matches_2to1 = []
for m,n in matches_2to1:
    if m.distance < 0.9 * n.distance:
        good_matches_2to1.append(m)
        
# Symmetric test
sym_matches = []
for m1 in good_matches_1to2:
    for m2 in good_matches_2to1:
        if m1.queryIdx == m2.trainIdx and m2.queryIdx == m1.trainIdx:
            sym_matches.append(m1)
            break
        
flann_matches_knn = [[m] for m in sym_matches] 
        
flann_matched_img = cv2.drawMatchesKnn(img1, kp1, img2, kp2, flann_matches_knn, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

cv2.imshow("FLANN Matched Points", flann_matched_img)
cv2.waitKey(10000)
cv2.destroyAllWindows()
